# PC-FNO3D Grand Tunis v4 — contrôle du branchement des capteurs (chemin de l'OS6) (Colab, A100)

Contrôle demandé par Ines le 11/11 : **aucun entraînement, aucune lecture du test**. Test absurde (+15 m/s sur u aux 50 capteurs simulés et, en témoin, aux aéroports), chemin des capteurs jusqu'au tenseur d'entrée du réseau, et (national, D) capteurs présents dans 5 lots d'entraînement. Sortie : `<run>/controle_capteurs.json`, journaux `journaux/controle_capteurs_*.log`.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "3f2e442453d3bdb992ca2d9bc5acc4d0ed6781d7"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. Cache

In [ ]:
# cache : les 3 parties du Drive -> disque local ; index fusionne
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (GT / f"cache_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())

# la grille complete (aneo_grid_points.csv) est dans jeu.zip
if not (GT / "best_dso_v7.pt").exists():
    raise FileNotFoundError("deposer best_dso_v7.pt (poids v7) dans MyDrive/pcfno3d_grand_tunis/")
lancer("controle", "python -u -m pcfno3d -c grand_tunis_v4_dso_colab check")

## 3. Contrôle, puis fin de la session

In [ ]:
import shlex
from google.colab import runtime
import traceback
RUN = GT / "runs" / "grand_tunis_v4_dso"
try:
    lancer("controle_capteurs_gt_v4",
           f"python -u scripts/controle_capteurs.py -c grand_tunis_v4_dso_colab --run {shlex.quote(str(RUN))} "
           "--set inference.w_continuite=true --instants 3 --n-capteurs 50 --delta 15 --rayon-km 5")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()